# Corazón v2 — E5: las 100 radiografías para la sesión de fotos

Solo copia archivos: **no usa GPU** y tarda unos minutos.

**Antes de correr (panel derecho de Kaggle):**
1. **Accelerator:** None (sin GPU: no gasta la cuota).
2. **Internet:** ON (clona el código de GitHub).
3. **Add Input:**
   - **Your Work → Notebooks** → el notebook del día 1, **Version 1** (trae `data/subset.csv` con la partición);
   - **Datasets** → `nih-chest-xrays/data` (las radiografías originales de 1024 px).
4. **Save Version → Save & Run All (Commit)**.

**Qué produce:** `e5_seleccion.zip` en el Output, con `seleccion.csv` y `png/`: 100 placas del test oficial, 25 por
grupo (PA sin cardiomegalia, PA con, AP sin, AP con), en orden mezclado. Se descarga y se descomprime en la laptop.

In [ ]:
import glob, os, shutil, subprocess, sys
W = "/kaggle/working"
REPO = f"{W}/repo"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/loupv99/corazon-v2", REPO], check=True)
os.chdir(REPO)
print("código:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

def buscar(rel):   # sin recorrer /kaggle/input entero (con NIH montado tarda minutos)
    for prof in range(5):
        r = sorted(glob.glob("/kaggle/input/" + "*/" * prof + rel))
        if r:
            return r[0]
    return ""

SUBSET = buscar("data/subset.csv")
NIH_CSV = buscar("Data_Entry_2017*.csv")
assert SUBSET, "Falta el Output del notebook del día 1 (Version 1) como input"
assert NIH_CSV, "Falta el dataset nih-chest-xrays/data como input"
print("subset:", SUBSET)
print("NIH   :", os.path.dirname(NIH_CSV))

In [ ]:
OUT = f"{W}/e5_seleccion"
subprocess.run([sys.executable, "-m", "src.fotos", "seleccionar", "--subset", SUBSET,
                "--nih-root", os.path.dirname(NIH_CSV), "--out", OUT], check=True)
import pandas as pd
sel = pd.read_csv(f"{OUT}/seleccion.csv")
print(sel.groupby(["view", "cardio"]).size())
print(len(os.listdir(f"{OUT}/png")), "PNG")
shutil.make_archive(f"{W}/e5_seleccion", "zip", OUT)
print("zip:", round(os.path.getsize(f"{W}/e5_seleccion.zip") / 1e6, 1), "MB")

In [ ]:
# Vista previa de la lámina que se fotografía
import cv2
from IPython.display import Image, display
sys.path.insert(0, REPO)
from src import fotos
f = fotos.leer_seleccion(OUT)[0]
cv2.imwrite(f"{W}/lamina_ejemplo.png", fotos.lamina(fotos.original(OUT, f), 0, "1/100"))
display(Image(f"{W}/lamina_ejemplo.png", width=800))
shutil.rmtree(OUT)   # queda solo el zip